# Chapter 6 &mdash; Minimization by Frames: $k$-Distinguishability

**Concept 8 of the Chapter 6 decomposition:** *DFA Minimization by Frames: $k$-Distinguishability*

Mark pairs 0-distinguishable if their finality differs, then propagate backwards frame by frame.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Frames-K-Distinguishability/Concept-Frames-K-Distinguishability.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The minimization algorithm fills a table of **distinguishability distances** over
unordered state pairs.

* **Frame 0:** a pair is **0-distinguishable** if exactly one of the two is final &mdash;
  the empty string separates them.
* **Frame $k{+}1$:** a pair $(p,q)$ becomes $(k{+}1)$-distinguishable if some symbol
  $a$ sends it to a pair already marked at distance $\le k$.
* Pairs still unmarked when nothing changes are **equivalent**.

The distance recorded is the **length of the shortest separating string**, which is
why the whole thing converges in at most $|Q|-1$ frames.

## 2. Definitions

### The machine to minimize

In [ ]:
D = md2mc('''DFA
IF : 0 -> A
IF : 1 -> B
A  : 0 -> IF
A  : 1 -> C
B  : 0 -> C
B  : 1 -> IF
C  : 0 -> B
C  : 1 -> A
''')

### The frame computation, written out

In [ ]:
def frames(D):
    qs = sorted(D["Q"])
    pairs = [(a, b) for i, a in enumerate(qs) for b in qs[i+1:]]
    dist = {p: (0 if (p[0] in D["F"]) != (p[1] in D["F"]) else -1) for p in pairs}
    def key(x, y): return (x, y) if (x, y) in dist else (y, x)
    k, log = 0, [dict(dist)]
    while True:
        changed = False
        for (p, q) in pairs:
            if dist[(p, q)] != -1: continue
            for a in sorted(D["Sigma"]):
                s, t = step_dfa(D, p, a), step_dfa(D, q, a)
                if s == t: continue
                if dist[key(s, t)] != -1 and dist[key(s, t)] <= k:
                    dist[(p, q)] = k + 1; changed = True; break
        k += 1; log.append(dict(dist))
        if not changed: return dist, log

### A separating string, to check each distance

In [ ]:
from itertools import product
def separator(D, p, q, maxlen):
    for k in range(maxlen + 1):
        for pr in product(sorted(D["Sigma"]), repeat=k):
            s = ''.join(pr)
            if (run_dfa_h(D, s, p) in D["F"]) != (run_dfa_h(D, s, q) in D["F"]):
                return s
    return None

## 3. Tests

Frame by frame, watch the marks propagate.

In [ ]:
dist, log = frames(D)
qs = sorted(D["Q"])
for i, snap in enumerate(log):
    marked = sorted(k for k, v in snap.items() if v != -1)
    print("after frame %d : %d pairs marked  %s" % (i, len(marked), marked))

Every recorded distance **is** the length of the shortest separating string.

In [ ]:
for (p, q), d in sorted(dist.items()):
    s = separator(D, p, q, len(D["Q"]))
    print("(%-3s,%-3s) distance %2s  shortest separator %r"
          % (p, q, d, s))
    if d == -1: assert s is None
    else:       assert s is not None and len(s) == d
print("\nevery distance verified against an explicit separating string")

Pairs left at $-1$ are the equivalent ones.

In [ ]:
eq = [k for k, v in dist.items() if v == -1]
print("equivalent pairs :", eq)
print("states %d - merged %d = %d ; min_dfa gives %d"
      % (len(D["Q"]), len(eq), len(D["Q"]) - len(eq), len(min_dfa(D)["Q"])))

Jove's own `fixptDist` computes the same table; `chatty=True` prints the frames.

In [ ]:
ht = {(a, b): (0 if (a in D["F"]) != (b in D["F"]) else -1)
      for i, a in enumerate(sorted(D["Q"])) for b in sorted(D["Q"])[i+1:]}
out = fixptDist(D, dict(ht))
print("\nfixptDist result :", dict(sorted(out.items())))

## 4. Exercises


1. Run `fixptDist(D, ht, chatty=True)` and read the frames it prints.
2. Why can the algorithm stop as soon as one frame changes nothing?
3. Give a DFA needing exactly $|Q|-1$ frames. (Hint: a long chain.)

In [ ]:
# Your work for the exercises above.